# Rerankers
> Score, reorder, and pipeline multiple reranking strategies.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `ScoreReranker`, `CrossEncoderReranker`, `FlashRankReranker`, `CohereReranker`, `RoundRobinReranker`, `RerankerPipeline`, `AsyncCrossEncoderReranker` |
| Difficulty | Intermediate |

Rerankers refine initial retrieval results by applying more expensive scoring
models. Anchor provides several strategies that can be combined into a pipeline.

**Time:** ~10 minutes

## Setup

In [ ]:
from anchor.retrieval import (
    ScoreReranker,
    CrossEncoderReranker,
    FlashRankReranker,
    CohereReranker,
    RoundRobinReranker,
    RerankerPipeline,
    AsyncCrossEncoderReranker,
)
from anchor.models import ContextItem, SourceType, QueryBundle

## Create Sample Results
Simulate initial retrieval results to rerank.

In [ ]:
def make_results():
    """Create mock retrieval results with varying scores."""
    data = [
        ("r-1", "Python is great for data science.", 0.85),
        ("r-2", "Rust ensures memory safety.", 0.72),
        ("r-3", "Go is designed for concurrency.", 0.68),
        ("r-4", "Python data pipelines with Pandas.", 0.91),
        ("r-5", "JavaScript powers the web.", 0.55),
        ("r-6", "Machine learning with Python and TensorFlow.", 0.88),
    ]
    return [
        ContextItem(id=did, content=text, source=SourceType.RETRIEVAL,
                   score=score, priority=5, token_count=10)
        for did, text, score in data
    ]

results = make_results()
print("Initial results (by original score):")
for r in results:
    print(f"  {r.id}: score={r.score:.2f}  {r.content[:40]}")

## ScoreReranker
Simply re-sorts results by their existing scores (useful as a baseline or
normalization step).

In [ ]:
score_reranker = ScoreReranker()
query = QueryBundle(query_str="Python data science")

reranked = score_reranker.rerank(query, make_results(), top_k=4)

print("ScoreReranker results:")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

## CrossEncoderReranker (Mocked)
In production, this calls a cross-encoder model (e.g., `sentence-transformers`).
Here we mock it with a simple scoring function.

In [ ]:
# Mock cross-encoder scoring function
def mock_cross_encoder(query: str, document: str) -> float:
    """Higher score if query terms appear in document."""
    query_terms = set(query.lower().split())
    doc_terms = set(document.lower().split())
    overlap = len(query_terms & doc_terms)
    return overlap / max(len(query_terms), 1)

cross_reranker = CrossEncoderReranker(score_fn=mock_cross_encoder)
reranked = cross_reranker.rerank(query, make_results(), top_k=4)

print("CrossEncoderReranker results:")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

## FlashRankReranker (Mocked)
FlashRank provides fast, lightweight reranking. We mock the scoring here.

In [ ]:
# Mock FlashRank scoring
def mock_flashrank(query: str, document: str) -> float:
    """Simple length-normalized overlap score."""
    q_words = set(query.lower().split())
    d_words = set(document.lower().split())
    return len(q_words & d_words) / max(len(d_words), 1)

flash_reranker = FlashRankReranker(score_fn=mock_flashrank)
reranked = flash_reranker.rerank(query, make_results(), top_k=4)

print("FlashRankReranker results:")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

## CohereReranker (Mocked)
The Cohere reranker calls the Cohere Rerank API. We mock it for demonstration.

In [ ]:
# Mock Cohere reranking
def mock_cohere(query: str, document: str) -> float:
    """Simulate Cohere-style relevance scoring."""
    q_set = set(query.lower().split())
    d_set = set(document.lower().split())
    overlap = len(q_set & d_set)
    return min(overlap * 0.3, 1.0)

cohere_reranker = CohereReranker(score_fn=mock_cohere)
reranked = cohere_reranker.rerank(query, make_results(), top_k=4)

print("CohereReranker results:")
for i, r in enumerate(reranked):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

## RoundRobinReranker
Interleaves results from multiple result sets in a round-robin fashion.
Useful for combining results from different retrieval strategies.

In [ ]:
rr_reranker = RoundRobinReranker()

# Simulate two different result sets
set_a = make_results()[:3]  # First 3
set_b = make_results()[3:]  # Last 3

interleaved = rr_reranker.interleave([set_a, set_b], top_k=5)

print("RoundRobinReranker interleaved results:")
for i, r in enumerate(interleaved):
    print(f"  [{i+1}] {r.id}  {r.content[:40]}")

## RerankerPipeline
Chain multiple rerankers into a sequential pipeline. Each stage refines
the output of the previous one.

In [ ]:
pipeline = RerankerPipeline(
    stages=[
        score_reranker,       # Stage 1: sort by score
        cross_reranker,       # Stage 2: cross-encoder refinement
    ]
)

final = pipeline.rerank(query, make_results(), top_k=3)

print("RerankerPipeline results:")
for i, r in enumerate(final):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

## AsyncCrossEncoderReranker (Mocked)
For async applications, use `AsyncCrossEncoderReranker` with `await arerank()`.

In [ ]:
import asyncio

async def mock_async_score(query: str, document: str) -> float:
    return mock_cross_encoder(query, document)

async_reranker = AsyncCrossEncoderReranker(score_fn=mock_async_score)

async def run_async_rerank():
    results = await async_reranker.arerank(query, make_results(), top_k=3)
    print("AsyncCrossEncoderReranker results:")
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r.score:.4f}  {r.content[:40]}")

asyncio.run(run_async_rerank())

## Key Takeaways
- `ScoreReranker` sorts by existing scores (baseline)
- `CrossEncoderReranker` applies a cross-encoder model for pairwise scoring
- `FlashRankReranker` and `CohereReranker` integrate external reranking services
- `RoundRobinReranker` interleaves results from multiple sources
- `RerankerPipeline` chains stages sequentially for multi-pass refinement
- `AsyncCrossEncoderReranker` provides async support via `await arerank()`

**Next:** [Routers](08_routers.ipynb)